<a href="https://colab.research.google.com/github/Vaibhav-Krishna-S/NavigateLabs/blob/main/RAGimplementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu transformers sentence-transformers


In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import faiss
import numpy as np


In [3]:
# Example notes
docs = [
    "Neuromorphic computing mimics the neural structure of the human brain.",
    "It uses spiking neural networks and asynchronous event-based processing.",
    "Neuromorphic chips are energy-efficient and ideal for edge AI.",
    "The term was introduced by Carver Mead in the 1980s."
]

# Chunking not needed here, but you can split longer text


In [4]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(docs, convert_to_tensor=False)

index = faiss.IndexFlatL2(doc_embeddings[0].shape[0])
index.add(np.array(doc_embeddings))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
def retrieve_relevant_docs(query, k=2):
    query_vec = embedder.encode([query])[0]
    D, I = index.search(np.array([query_vec]), k)
    return [docs[i] for i in I[0]]


In [6]:
generator = pipeline("text-generation", model="distilgpt2")

def generate_answer(query):
    context = retrieve_relevant_docs(query)
    prompt = "Context: " + " ".join(context) + "\nQuestion: " + query + "\nAnswer:"
    output = generator(prompt, max_length=100, do_sample=True)[0]['generated_text']
    return output


Device set to use cpu


In [7]:
question = "Who coined the term neuromorphic computing?"
print(generate_answer(question))


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context: Neuromorphic computing mimics the neural structure of the human brain. Neuromorphic chips are energy-efficient and ideal for edge AI.
Question: Who coined the term neuromorphic computing?
Answer: There is a real, scientific debate (with many experts), and it's difficult to define what brain we're doing with the concept, however, if these two people, like Watson, who use the term, don't know how long the term term is going to last before or
